<a href="https://colab.research.google.com/github/fdavenport/CIVE480A-Fall26/blob/main/Lab2_Analyzing_Historical_Temperature_Trends.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lab 2: Analyzing Historical Temperature Trends

In this lab, we will use Python to:
* work with gridded spatial climate data (called "rasters")
* explore global patterns of temperature change
* analyze regional patterns of climate change in Colorado
* calculate temperature trends and interpret their statistical significance

**Setting up**

We will use a new library in this lab, `xarray`, which is designed for working with multi-dimensional gridded data (like the latitude/longitude/time grids used in climate or weather datasets). We also import `numpy` and `matplotlib`, which we have used previously.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

The data files for this lab live in the `data/` folder of the course GitHub repository. As in Lab 1, we store the web address of that folder in a variable so we can reuse it each time we read a file.

In [ ]:
data_url = "https://raw.githubusercontent.com/fdavenport/CIVE480A-Fall26/main/data/"

# Part 1: Patterns of Global Temperature Change

In this section, we will analyze gridded global temperature data from NOAA's **Global Surface Temperature dataset**. This dataset combines temperature measurements from weather stations on land with sea surface temperature measurements into a single temperature dataset covering the globe from 1850 to present.

The values in this file are **annual temperature anomalies** in °C. These anomalies are the difference between the observed temperature for a given year compared to the average over the 1971-2000.

The NOAA temperature data is sorted in a **netCDF** file (`.nc`), a common binary file format for storing gridded scientific data. Because it is a binary file (not plain text), we can't read it directly from a URL the way we did with `pd.read_csv()`. Instead, we first download the file, and then open it with `xarray`.

In [ ]:
import urllib.request

In [ ]:
urllib.request.urlretrieve(data_url + "noaa_global_temperature_annual.nc", "noaa_global_temperature_annual.nc")

**Opening and exploring the dataset**

`xarray` organizes gridded data into a **Dataset**, made up of:
* **dimensions and coordinates** — here, `year`, `lat`, and `lon`
* **data variables** — here, `temperature`, which has values for every combination of year, latitude, and longitude


**Selecting data for a single location**

**Selecting data for a single year**

We can overlay country boundaries so the map is easier to interpret. Country boundaries are stored in a **GeoJSON** file, which we can read directly with the `geopandas` library (`geopandas` is built for working with maps and geographic boundaries).

In [ ]:
import geopandas as gpd

world_boundaries = gpd.read_file(
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"
)

In [ ]:
## add code to plot temperature data


world_boundaries.boundary.plot(ax=ax, color="black", linewidth=0.5)

plt.colorbar(mesh, ax=ax, label="Temperature anomaly (°C)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Global temperature anomaly, {year_to_plot}")
plt.show()

**Interactive map**

The code below uses `ipywidgets` to create an interactive slider so you can look at the map for any year in the dataset.

In [ ]:
import ipywidgets as widgets

In [ ]:
def plot_year(year):
    fig, ax = plt.subplots(figsize=(12, 6))
    mesh = ax.pcolormesh(global_temp["lon"], global_temp["lat"], global_temp["temperature"].sel(year=year),
                          cmap="RdBu_r", vmin=-3, vmax=3, shading="auto")
    world_boundaries.boundary.plot(ax=ax, color="black", linewidth=0.5)
    plt.colorbar(mesh, ax=ax, label="Temperature anomaly (°C)")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"Global temperature anomaly, {year}")
    plt.show()

widgets.interact(plot_year, year=widgets.IntSlider(int(global_temp["year"].min()), int(global_temp["year"].min()), 
                                                   int(global_temp["year"].max()), 1, layout=widgets.Layout(width='700px')));

<div style="color:#1a73e8;">

**1a)** Based on the maps above, describe the patterns of global temperature change. Are there regions that appear to be warming faster than others? How has the pace of global temperature change varied over the historical record?

</div>

*add answer here*

**Calculating the global mean temperature timeseries**

We will take the mean of values in each year to calculate average global temperature. However, our grid cells are not all the same area. Because lines of longitude converge toward the poles, grid cells near the poles cover much less area on the Earth's surface than grid cells near the equator.

To account for this, we will weight each grid cell temperature value by the grid cell area, which is proportional to the cosine of it's latitude. We will use the built-in `.weighted()` function in `xarray`. 

**Ploting the global mean temperature time series**

We will make a plot of global temperature vs. year

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))



ax.set_xlabel("Year")
ax.set_ylabel("Temperature anomaly (°C)")
ax.set_title("Global mean temperature anomaly")
plt.show()

**Linear trends and statistical significance**

To describe how temperature has changed over time, we can fit a **linear trend** — a straight line that best describes the relationship between year (x) and temperature (y). The slope of this line tells us the rate of change (e.g., °C per year).

We will use the function `scipy.stats.linregress()`, which fits a line and returns several useful values:
* `slope` and `intercept` — the fitted line is `y = slope * x + intercept`
* `rvalue` — the correlation coefficient (how well the line fits the data)
* `pvalue` — the p-value for a hypothesis test of whether the slope is significantly different from zero


In [ ]:
from scipy import stats

**Calculating trends over different time periods**

We will calculate the linear trend in the global mean temperature over three different time periods:
* the full record: 1895–2025
* the last 30 years: 1996–2025
* the last 15 years: 2011–2025


Calculate trend for full period: 

In [ ]:
global_full_record = global_mean_temp.sel(year=slice(1895, 2025))
global_full_trend = stats.linregress(global_full_record["year"], global_full_record)

In [ ]:
print("1895-2025 trend:", round(global_full_trend.slope * 10, 3), "degC / decade")
print("p-value:", round(global_full_trend.pvalue, 4))

<div style="color:#1a73e8;">

**1b)** Calculate the linear trend for 1996–2025 (the last 30 years) in the global mean temperature, following the same steps as above. Print the slope (°C/decade) and p-value for the trend.

</div>

In [ ]:
# TO DO: calculate the 1996-2025 trend


<div style="color:#1a73e8;">

**1c)** Calculate the linear trend for 2011–2025 (the last 15 years) in the global mean temperature, following the same steps as above. Print the slope (°C/decade) and p-value.

</div>

In [ ]:
# TO DO: calculate the 2011-2025 trend


**Plotting the trends together**

Now that we have all three trends, let's compare them on a single time series graph: the observed global mean temperature, with each period's fitted trend line overlaid. The observed data and the full-record (1895–2025) trend line are plotted for you below.

<div style="color:#1a73e8;">

**1d)** Add the 1996–2025 trend line and the 2011–2025 trend line to the plot below, each with a label for the legend.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(global_mean_temp["year"], global_mean_temp, "o-", color="gray", markersize=4, label="observed")
ax.plot(global_full_record["year"], global_full_trend.intercept + global_full_trend.slope * global_full_record["year"], label="1895-2025 trend")

# TO DO: add the 1996-2025 and 2011-2025 trend lines


ax.set_xlabel("Year")
ax.set_ylabel("Temperature anomaly (°C)")
ax.set_title("Global mean temperature trends")
ax.legend()
plt.show()

<div style="color:#1a73e8;">

**1e)** Compare the slope and statistical significance of the three global trends you calculated. How does the most recent (15-year) trend compare to the longer-term trends? Based on this comparison, what would you consider an appropriate length of time for a climate trend analysis? 

</div>

*add answer here*

# Part 2: Analyzing Temperature Change in Fort Collins

The NOAA Global Temperature data that we used in Part 1 has a fairly coarse resolution (5° x 5° grid cells, which are roughly 500 km on a side), which is too coarse to say much about temperature change in a specific place like Fort Collins. In this section, we will use a higher-resolution dataset called **PRISM** (Parameter-elevation Regressions on Independent Slopes Model), produced by a research group at Oregon State University: <https://prism.oregonstate.edu/>.

PRISM combines weather station observations with elevation and terrain information to produce gridded climate data at ~4 km resolution across the United States. The file `PRISM_tmean_annual_CO_1895_2025.nc` contains annual mean temperature (°C) for Colorado from 1895–2025.

In [ ]:
urllib.request.urlretrieve(data_url + "PRISM_tmean_annual_CO_1895_2025.nc", "PRISM_tmean_annual_CO_1895_2025.nc")

<div style="color:#1a73e8;">

**2a)** Read in `PRISM_tmean_annual_CO_1895_2025.nc` using `xarray`. Save it to a variable called `co_temp`.

</div>

In [ ]:
# TO DO: read in PRISM_tmean_annual_CO_1895_2025.nc as co_temp


<div style="color:#1a73e8;">

**2b)** Selecting data for Fort Collins

Fort Collins, CO is located at approximately 40.585°N, 105.084°W. Select the temperature time series for the grid cell nearest Fort Collins (using `.sel(..., method="nearest")`.

</div>

In [ ]:
fort_collins_lat, fort_collins_lon = 40.585, -105.084

# TO DO: select the temperature time series for the grid cell nearest Fort Collins


<div style="color:#1a73e8;">
    
**2c)** Plot a time series of the Fort Collins temperature data
</div>

In [ ]:
# TO DO: plot time series of Fort Collins Temperature


**Calculating trends over different time periods**

Now you will calculate the same three trends for Fort Collins that we calculated for the global mean in Part 1: the full record (1895–2025), the last 30 years (1996–2025), and the last 15 years (2011–2025). The calculation for the full record is done for you below.

In [ ]:
from scipy import stats

fc_temp = co_temp["temperature"].sel(y=fort_collins_lat, x=fort_collins_lon, method="nearest")

full_record = fc_temp.sel(year=slice(1895, 2025))
full_trend = stats.linregress(full_record["year"], full_record)

print("1895-2025 trend:", round(full_trend.slope * 10, 3), "degC / decade")
print("p-value:", round(full_trend.pvalue, 4))

<div style="color:#1a73e8;">

**2d)** Calculate the linear trend for 1996–2025 (the last 30 years). Print the slope (°C/decade) and p-value.

</div>

In [ ]:
# TO DO: calculate the 1996-2025 trend


<div style="color:#1a73e8;">

**2e)** Calculate the linear trend for 2011–2025 (the last 15 years). Print the slope (°C/decade) and p-value.

</div>

In [ ]:
# TO DO: calculate the 2011-2025 trend


**Plotting the trends together**

Now that we have all three trends, let's compare them on a single time series graph: the observed Fort Collins temperatures, with each period's fitted trend line overlaid. The observed data and the full-record (1895–2025) trend line are plotted for you below.

<div style="color:#1a73e8;">

**2f)** Add the 1996–2025 trend line and the 2011–2025 trend line to the plot below, each with a label for the legend.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fc_temp["year"], fc_temp, "o-", color="gray", markersize=4, label="observed")
ax.plot(full_record["year"], full_trend.intercept + full_trend.slope * full_record["year"], label="1895-2025 trend")

# TO DO: add the 1996-2025 and 2011-2025 trend lines


ax.set_xlabel("Year")
ax.set_ylabel("Mean annual temperature (°C)")
ax.set_title("Fort Collins temperature trends")
ax.legend()
plt.show()

<div style="color:#1a73e8;">

**2g)** Compare the time series and the slope and statistical significance of the three Fort Collins trends you calculated to the three global trends from Part 1. What similarities or differences do you notice? What would you consider an appropriate time period when calculated temperature trends at a specific location? 

</div>

*add answer here*

# Part 3: Analyzing Temperature Change across Colorado

Now you will use the same PRISM dataset to look at spatial patterns of temperature change across the whole state of Colorado, rather than a single location.

The code below calculates average temperature in each grid cell average over the early 20th century (1900–1950).

In [ ]:
early_period = co_temp["temperature"].sel(year=slice(1900, 1950)).mean(dim="year")
early_period

<div style="color:#1a73e8;">

**3a)** Repeat this calculation, but for a **recent period** using the years 2006–2025.

</div>

In [ ]:
# TO DO: calculate recent_period (2006-2025 mean at every grid cell)


**Comparing the two periods side by side**

Before calculating the difference between the two periods, you will plot them side by side on the same color scale so we can compare them directly. To make the maps easier to interpret, we'll also add Colorado county boundaries and a few city locations for reference. The panel for the early period (1900–1950), including the boundaries and cities, is plotted for you below.

In [ ]:
## read county boundaries and define city locations to use for plotting

counties = gpd.read_file("https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json")
co_counties = counties[counties["STATE"] == "08"] ## subset for Colorado counties

cities = {
    "Fort Collins": (40.585, -105.084),
    "Denver": (39.739, -104.990),
    "Colorado Springs": (38.833, -104.821),
    "Grand Junction": (39.064, -108.550),
    "Durango": (37.276, -107.881),
}

<div style="color:#1a73e8;">

**3b)** Plot the recent-period (2006–2025) map in the second panel below, using the same colormap and color scale (`vmin`/`vmax`) as the first panel, with the title `"2006-2025 average"`. Add the county boundaries and city markers to this panel too, just as we did for the first panel.

</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
vmin = min(early_period.min(), recent_period.min())
vmax = max(early_period.max(), recent_period.max())

## plot early period data: 
mesh = axes[0].pcolormesh(co_temp["x"], co_temp["y"], early_period, cmap="viridis", vmin=vmin, vmax=vmax, shading="auto")
axes[0].set_title("1900-1950 average")


# TO DO: plot recent_period in axes[1] (same cmap, vmin, vmax),



## we will use a loop to add the county/city formatting to both axes
for ax in axes:
    for name, (city_lat, city_lon) in cities.items():
        ax.plot(city_lon, city_lat, "o", color="blue", markersize=4)
        ax.annotate(name, (city_lon, city_lat), textcoords="offset points", xytext=(4, 4), fontsize=8)
    co_counties.boundary.plot(ax=ax, color="black", linewidth=0.4)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

plt.colorbar(mesh, ax=axes, label="Mean annual temperature (°C)")
plt.show()

**Calculating temperature change**

Subtract `early_period` from `recent_period` to calculate the temperature change at every grid cell.

In [ ]:
temp_change = recent_period - early_period
temp_change

<div style="color:#1a73e8;">

**3c) Mapping temperature change**

The figure is set up for you below, already including the county boundaries and city markers from the map above. Add a line of code to plot `temp_change` with `plt.pcolormesh()`. 

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# TO DO: add a pcolormesh call to plot temp_change, and save it as `mesh`


co_counties.boundary.plot(ax=ax, color="black", linewidth=0.4)
for name, (city_lat, city_lon) in cities.items():
    ax.plot(city_lon, city_lat, "o", color="blue", markersize=4)
    ax.annotate(name, (city_lon, city_lat), textcoords="offset points", xytext=(4, 4), fontsize=8)

plt.colorbar(mesh, ax=ax, label="Temperature change (°C)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Colorado temperature change: 2006-2025 vs. 1900-1950")
plt.show()

<div style="color:#1a73e8;">

**3d)** Describe the spatial pattern of temperature change in Colorado shown in the map above. What regions appear to be warming faster than others?

</div>

*add answer here*

### Exporting this notebook as an HTML file

When you're done, convert your work to an HTML file using the code below. The HTML file will preserve your code, text, and output as it appears. You can then submit the HTML file on Canvas.

Before running it, make sure you have run all of the cells you want included so their output shows up. The easiest way is to select "Restart session and run all" under the Runtime dropdown menu at the top of Colab. When it finishes in Colab, the HTML file will download automatically.

In [ ]:
# 1. make sure this matches your notebook's filename (shown at the top-left of the Colab window)
notebook_filename = "Lab2_Analyzing_Historical_Temperature_Trends.ipynb"

# 2. if running in Google Colab, the .ipynb file is not on the runtime's disk, so we
#    grab the current notebook from Colab and save it before converting
try:
    import json
    from google.colab import _message
    notebook_json = _message.blocking_request("get_ipynb", request="", timeout_sec=60)["ipynb"]
    with open(notebook_filename, "w") as f:
        json.dump(notebook_json, f)
    in_colab = True
except ImportError:
    in_colab = False

# 3. convert the notebook to an HTML file
!jupyter nbconvert --to html "{notebook_filename}"

# 4. if running in Google Colab, automatically download the HTML file
if in_colab:
    from google.colab import files
    files.download(notebook_filename.replace(".ipynb", ".html"))